In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from catboost import CatBoostClassifier

import matplotlib.pyplot as plt
import seaborn as sns

import joblib

In [3]:
data = pd.read_csv("data.csv")

print(data.head())
print(data.shape)

     Branch  FacilityAmount Granted Date  Tenor  Effective Rate  FlatRate  \
0  GODAGAMA        288330.0     4/9/2018     36            25.0     20.06   
1  GODAGAMA        311430.0     4/9/2018     36            25.0     20.33   
2  GODAGAMA       2200000.0     4/9/2018     60            20.0     12.43   
3  GODAGAMA        232330.0    4/10/2018     48            25.0     19.95   
4  GODAGAMA        187530.0    4/10/2018     36            25.0     21.02   

  Type of Rental Paid SchemeType  Prepayment  NetRental  ...  Advance  \
0             MONTHLY     NORMAL           0    12828.0  ...      0.0   
1             MONTHLY     NORMAL           0    13927.0  ...      0.0   
2             MONTHLY     NORMAL           0    59459.0  ...      0.0   
3             MONTHLY     NORMAL           0     8703.0  ...      0.0   
4             MONTHLY     NORMAL           0     8494.0  ...      0.0   

   AdvanceRental  AdvanceSundry  AdvanceOther  Equipment Type  \
0            0.0              0  

In [4]:
data.columns = data.columns.str.strip()

In [5]:
print(data.isnull().sum())

Branch                         0
FacilityAmount                 0
Granted Date                  40
Tenor                          0
Effective Rate                 0
FlatRate                     293
Type of Rental Paid            0
SchemeType                     0
Prepayment                     0
NetRental                    293
DownPayment                  293
No of Rental in arrears        0
Age                            0
ArrearsCapital                 0
ArrearsInterest                0
ArrearsVat                     0
ArrearsOD                      0
ArrearsOther                   0
ArrearsInsu                    0
ArrearsSundry                  0
Advance                        0
AdvanceRental                  0
AdvanceSundry                  0
AdvanceOther                   0
Equipment Type              1127
Status                         0
Last Receipt Paid Amount    1173
NET-OUTSTANDING                0
ArrearsInsuEasyPay             0
arrears_intensity            231
dtype: int

In [6]:
data = data.fillna(data.median(numeric_only=True))

In [26]:
features = [
'FacilityAmount',
'Tenor',
'Effective Rate',
'FlatRate',
'NetRental',
'DownPayment',
'No of Rental in arrears',
'Age',
'ArrearsCapital',
'ArrearsInterest',
'ArrearsVat',
'ArrearsOD',
'ArrearsOther',
'ArrearsInsu',
'ArrearsSundry',
'Advance',
'AdvanceRental',
'AdvanceSundry',
'AdvanceOther',
'Last Receipt Paid Amount',
'NET-OUTSTANDING',
'ArrearsInsuEasyPay',
'arrears_intensity'
]

X = data[features]
y = data["Status"]

In [ ]:
y = data["Status"].str.strip()

y = y.map({
    "Current Running": 0,
    "Normal Settlement Completed": 0,
    "Early Settlement Completed": 0,
    "Write Off": 1,
    "REPOSSESSION AND SOLD": 1,
    "Rescheduled": 1,
    "Activated / Not Printed": 1,
    "Initiated / Not Activated": 1
})

# Drop rows where y is NaN (in case of unexpected values)
mask = y.notna()
X = X[mask]
y = y[mask]

After mapping, y unique: [0 1]
Len y: 188866
Len X before mask: 188866
After drop, len X: 188866 len y: 188866


In [28]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [29]:
X_train["arrears_ratio"] = X_train["ArrearsCapital"] / (X_train["FacilityAmount"] + 1)
X_test["arrears_ratio"] = X_test["ArrearsCapital"] / (X_test["FacilityAmount"] + 1)

X_train["loan_age_ratio"] = X_train["Tenor"] / (X_train["Age"] + 1)
X_test["loan_age_ratio"] = X_test["Tenor"] / (X_test["Age"] + 1)

In [30]:
model = CatBoostClassifier(

    iterations=1000,
    learning_rate=0.03,
    depth=8,
    eval_metric="Accuracy",
    random_seed=42,
    verbose=100

)

model.fit(X_train, y_train)

0:	learn: 0.9917137	total: 366ms	remaining: 6m 5s
100:	learn: 0.9924351	total: 7.31s	remaining: 1m 5s
200:	learn: 0.9929182	total: 13.1s	remaining: 52s
300:	learn: 0.9931300	total: 18.2s	remaining: 42.3s
400:	learn: 0.9934609	total: 23.8s	remaining: 35.5s
500:	learn: 0.9938316	total: 29.8s	remaining: 29.7s
600:	learn: 0.9941162	total: 35.7s	remaining: 23.7s
700:	learn: 0.9943611	total: 41.8s	remaining: 17.8s
800:	learn: 0.9945199	total: 48.2s	remaining: 12s
900:	learn: 0.9946126	total: 55.1s	remaining: 6.05s
999:	learn: 0.9947648	total: 1m 3s	remaining: 0us


CatBoostClassifier(depth=8, eval_metric='Accuracy', iterations=1000, learning_rate=0.03, random_seed=42, verbose=100)

["'Early Settlement Completed'", "'Write Off'", "'Normal Settlement Completed'", "'Current Running'", "'REPOSSESSION AND SOLD'", "'Rescheduled'", "'Activated / Not Printed'", "'Initiated / Not Activated'"]
